In [1]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
from collections import Counter
import torch
from scipy.stats import randint, uniform

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

from torch.utils.data import TensorDataset, DataLoader

# Modelos
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier # Mucho más rápido para datos densos VAE que GradientBoostingClassifier

warnings.filterwarnings('ignore')
print("✅ Librerías cargadas correctamente.")

✅ Librerías cargadas correctamente.


In [6]:

# 1. Configurar dispositivo y variables
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

base_dir = r"C:\Users\Usuario\Documents\Workspace\Mirage\TFG\dataset\dataset_finales\VAE"
batch_size = 32  # Asegurar definición de batch_size

dataset_type = "descripciones"

# 2. Rutas del dataset VAE
if dataset_type == "codigos":
    train_path = os.path.join(base_dir, "codigos_train_VAE.csv")
    test_path = os.path.join(base_dir, "codigos_test_VAE.csv")
elif dataset_type == "descripciones":
    train_path = os.path.join(base_dir, "descripciones_train_VAE.csv")
    test_path = os.path.join(base_dir, "descripciones_test_VAE.csv")
else:  # combinado
    train_path = os.path.join(base_dir, "combinado_train_VAE.csv")
    test_path = os.path.join(base_dir, "combinado_test_VAE.csv")

df_train = pd.read_csv(train_path, sep="|")
df_test = pd.read_csv(test_path, sep="|")

# 3. Filtrar por códigos objetivo
codes = sorted(["F20", "F21", "F22", "F23", "F25", "F29", "F60.1"])
df_train = df_train[df_train["DIAG PSQ"].isin(codes)].copy()
df_test = df_test[df_test["DIAG PSQ"].isin(codes)].copy()

# 4. Filtrado estricto de características numéricas del VAE
target_col = "DIAG PSQ"
columnas_basura = [target_col, "id", "text", "Unnamed: 0", "label", "Origen"]

# Excluir basura y columnas que empiecen por 'Diag'/'diag'
feature_cols = [
    col for col in df_train.columns 
    if col not in columnas_basura and not col.lower().startswith("diag")
]

# Seleccionar SOLO columnas numéricas y reemplazar NaNs por 0
X_train_df = df_train[feature_cols].select_dtypes(include=[np.number]).fillna(0)
X_test_df = df_test[feature_cols].select_dtypes(include=[np.number]).fillna(0)

X_train = X_train_df.values
X_test = X_test_df.values

# 5. Codificar etiquetas alineadas
cat_type = pd.CategoricalDtype(categories=codes, ordered=True)
y_train = df_train[target_col].astype(cat_type).cat.codes.values
y_test = df_test[target_col].astype(cat_type).cat.codes.values

num_classes = len(codes)
input_dim = X_train.shape[1]

# 6. Escalado de características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------------------------
# OPCIÓN A: Para modelos Scikit-Learn, LightGBM y XGBoost
# Usar: X_train_scaled, y_train, X_test_scaled, y_test
# -------------------------------------------------------------------

# -------------------------------------------------------------------
# OPCIÓN B: Para Redes Neuronales / PyTorch / BERT Embeddings VAE
# Usar: train_loader y test_loader
# -------------------------------------------------------------------
seed = int(time.time_ns() % (2**32))
torch.manual_seed(seed)
np.random.seed(seed)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\n✅ Datos VAE cargados con éxito:")
print(f"Dimensión de características numéricas: {input_dim}")
print(f"Número de clases: {num_classes}")
print("Distribución Entrenamiento:", Counter(y_train))
print("Distribución Evaluación:", Counter(y_test))

Usando dispositivo: cuda

✅ Datos VAE cargados con éxito:
Dimensión de características numéricas: 768
Número de clases: 7
Distribución Entrenamiento: Counter({np.int8(0): 400, np.int8(2): 291, np.int8(5): 160, np.int8(4): 118, np.int8(3): 75, np.int8(6): 30, np.int8(1): 18})
Distribución Evaluación: Counter({np.int8(0): 100, np.int8(2): 54, np.int8(5): 30, np.int8(4): 22, np.int8(3): 14, np.int8(6): 3, np.int8(1): 1})


In [7]:
print("🚀 Iniciando Randomized Search para LIGHTGBM...")

# Definición de distribuciones en lugar de listas fijas
param_dist_lgbm = {
    'n_estimators': [100, 200, 300],
    'learning_rate': uniform(0.01, 0.1),
    'max_depth': [3, 5, 7, -1],
    'num_leaves': randint(15, 63),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4)
}

# Usar StratifiedKFold para mantener la proporción de clases escasas
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

rand_lgbm = RandomizedSearchCV(
    estimator=LGBMClassifier(random_state=seed, verbose=-1, n_jobs=1),
    param_distributions=param_dist_lgbm,
    n_iter=50, 
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=-1,
    random_state=seed,
    verbose=1
)

t0 = time.time()
rand_lgbm.fit(X_train_scaled, y_train) # Corregida la variable a X_train_scaled
t_total = time.time() - t0

print(f"\n⏱️ Búsqueda completada en {t_total:.2f} segundos.")
print("🏆 MEJORES HIPERPARÁMETROS (LightGBM):")
print(rand_lgbm.best_params_)

best_lgbm = rand_lgbm.best_estimator_
y_pred_lgbm = best_lgbm.predict(X_test_scaled)

print("\n📊 REPORTE DE CLASIFICACIÓN EN TEST (LightGBM):")
print(classification_report(y_test, y_pred_lgbm, target_names=codes))

🚀 Iniciando Randomized Search para LIGHTGBM...
Fitting 5 folds for each of 50 candidates, totalling 250 fits

⏱️ Búsqueda completada en 1781.81 segundos.
🏆 MEJORES HIPERPARÁMETROS (LightGBM):
{'colsample_bytree': np.float64(0.7491932439908457), 'learning_rate': np.float64(0.07638785688389557), 'max_depth': 3, 'n_estimators': 200, 'num_leaves': 45, 'subsample': np.float64(0.9265884307357256)}

📊 REPORTE DE CLASIFICACIÓN EN TEST (LightGBM):
              precision    recall  f1-score   support

         F20       0.52      0.77      0.62       100
         F21       0.00      0.00      0.00         1
         F22       0.36      0.33      0.35        54
         F23       0.00      0.00      0.00        14
         F25       0.25      0.09      0.13        22
         F29       0.31      0.17      0.22        30
       F60.1       0.00      0.00      0.00         3

    accuracy                           0.46       224
   macro avg       0.21      0.19      0.19       224
weighted avg   

In [ ]:
print("🚀 Iniciando Randomized Search para XGBOOST...")

param_dist_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': uniform(0.01, 0.1),
    'max_depth': [3, 5, 7],
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 0.3)
}

rand_xgb = RandomizedSearchCV(
    estimator=XGBClassifier(
        random_state=seed, 
        eval_metric='mlogloss', 
        tree_method='hist', # CLAVE: Acelera drásticamente embeddings VAE continuos
        n_jobs=1
    ),
    param_distributions=param_dist_xgb,
    n_iter=20,
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=-1,
    random_state=seed,
    verbose=1
)

t0 = time.time()
rand_xgb.fit(X_train_scaled, y_train) # Corregida la variable a X_train_scaled
t_total = time.time() - t0

print(f"\n⏱️ Búsqueda completada en {t_total:.2f} segundos.")
print("🏆 MEJORES HIPERPARÁMETROS (XGBoost):")
print(rand_xgb.best_params_)

best_xgb = rand_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test_scaled)

print("\n📊 REPORTE DE CLASIFICACIÓN EN TEST (XGBoost):")
print(classification_report(y_test, y_pred_xgb, target_names=codes))

🚀 Iniciando Randomized Search para XGBOOST...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [ ]:
print("🚀 Iniciando Randomized Search para HIST GRADIENT BOOSTING...")

# HistGradientBoostingClassifier es equivalente a LightGBM en Scikit-Learn y procesa matrices densas x100 más rápido
param_dist_hgb = {
    'max_iter': [100, 200, 300],
    'learning_rate': uniform(0.01, 0.1),
    'max_depth': [3, 5, 7, None],
    'l2_regularization': uniform(0, 1.0)
}

rand_hgb = RandomizedSearchCV(
    estimator=HistGradientBoostingClassifier(random_state=seed),
    param_distributions=param_dist_hgb,
    n_iter=20,
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=-1,
    random_state=seed,
    verbose=1
)

t0 = time.time()
rand_hgb.fit(X_train_scaled, y_train)
t_total = time.time() - t0

print(f"\n⏱️ Búsqueda completada en {t_total:.2f} segundos.")
print("🏆 MEJORES HIPERPARÁMETROS (HistGradientBoosting):")
print(rand_hgb.best_params_)

best_hgb = rand_hgb.best_estimator_
y_pred_hgb = best_hgb.predict(X_test_scaled)

print("\n📊 REPORTE DE CLASIFICACIÓN EN TEST (HistGradientBoosting):")
print(classification_report(y_test, y_pred_hgb, target_names=codes))

🚀 Iniciando Grid Search para GRADIENT BOOSTING (Scikit-Learn)...
Fitting 5 folds for each of 144 candidates, totalling 720 fits


KeyboardInterrupt: 